# Variogram models & estimators

The variogram is the heart of kriging: it encodes *how fast the field de-correlates with distance*. This notebook
is a tour of the **theoretical models** geostatista ships, the two **empirical estimators**, and how to **choose a
model** with cross-validation. See the [kriging workflow](01_kriging_workflow.ipynb) for the end-to-end pipeline.

In [ ]:
%matplotlib inline
import numpy as np
from cleopatra.glyphs.primitives.line_glyph import LineGlyph
from geostatista import models

print("bounded (fittable) models :", sorted(models.BOUNDED_MODELS))
print("all model functions       :", sorted(models.MODELS))

## 1. The bounded models

Every bounded model is `gamma(h; nugget, sill, range)`: it starts at the **nugget** (a micro-scale discontinuity
at `h -> 0+`), rises with lag `h`, and levels off at the **sill**. They differ in shape:

- **spherical** reaches the sill *exactly* at the range;
- **exponential** approaches it asymptotically (practical range at `h = range`);
- **gaussian** is smooth and parabolic near the origin — good for very continuous fields;
- **matern** adds a smoothness parameter `nu` (here 1.5); note it uses a *different* range
  convention (the Stein `sqrt(2*nu)*h/range`, a 1/e scaling), so at the same nominal `range` its
  curve is not directly comparable to the three bounded models above.

In [ ]:
h = np.linspace(0.0, 1.5, 200)
nugget, sill, rng = 0.2, 1.0, 1.0

# One cleopatra LineGlyph draws all four model curves (2-D y -> one line per column).
curves = np.column_stack([models.MODELS[m](h, nugget, sill, rng)
                          for m in ["spherical", "exponential", "gaussian", "matern"]])
fig, ax, _ = LineGlyph(
    h, curves, figsize=(7, 4.5),
    title="Bounded variogram models (nugget=0.2, sill=1, range=1)",
).line(label=["spherical", "exponential", "gaussian", "matern"])

# sill and range guides, drawn as dotted cleopatra line segments
LineGlyph(np.array([0.0, h.max()]), np.array([sill, sill]), linestyle=":").line(ax=ax, color="grey")
LineGlyph(np.array([rng, rng]), np.array([0.0, sill]), linestyle=":").line(ax=ax, color="grey")
ax.set_xlabel("lag distance h")
ax.set_ylabel("semivariance γ(h)")
ax.legend()

## 2. The unbounded and pure-nugget models

Two more model functions are available (as functions, not as fittable models, because they have a different
parameterization):

- **power** — `gamma(h) = nugget + scale · h^exponent` with `0 < exponent < 2`: unbounded, for non-stationary
  fields with no sill;
- **nugget** — a flat `gamma(h) = nugget` for all `h > 0`: pure noise, no spatial structure.

In [ ]:
# power model — unbounded, one curve per exponent
powers = np.column_stack([models.power(h, 0.0, 1.0, e) for e in (0.5, 1.0, 1.5)])
fig, ax, _ = LineGlyph(h, powers, figsize=(7, 4),
                       title="power model (unbounded)").line(
    label=["exponent=0.5", "exponent=1.0", "exponent=1.5"])
ax.set_xlabel("h")
ax.set_ylabel("γ(h)")
ax.legend()

# pure-nugget model — flat, no spatial structure
fig, ax, _ = LineGlyph(h, models.nugget(h, 0.6), figsize=(7, 4),
                       title="pure-nugget model (no spatial structure)").line(color="#DC143C")
ax.set_xlabel("h")
ax.set_ylabel("γ(h)")
ax.set_ylim(0.0, 1.0)

## 3. Matheron vs Cressie estimators

The **empirical** variogram can be estimated two ways. The classic **Matheron** estimator averages squared
differences; the **Cressie** estimator is a robust alternative that down-weights outliers. On clean data they
agree; add a wild outlier and Matheron's cloud inflates while Cressie stays put.

In [ ]:
import geopandas as gpd
from shapely.geometry import Point
from geostatista import Samples

rng_ = np.random.default_rng(1)
xy = rng_.uniform(0.0, 100.0, (90, 2))
field = np.sin(xy[:, 0] / 20.0) * 8.0 + 20.0
field[0] += 60.0                         # inject one gross outlier
gdf = gpd.GeoDataFrame({"z": field}, geometry=[Point(*p) for p in xy], crs="EPSG:32633")
samples = Samples(gdf)

matheron = samples.variogram("z", n_lags=12, estimator="matheron").to_dataframe()
cressie = samples.variogram("z", n_lags=12, estimator="cressie").to_dataframe()

# Two cleopatra series (different markers) on one shared axis.
fig, ax, _ = LineGlyph(matheron["lag"].to_numpy(), matheron["semivariance"].to_numpy(),
                       marker="o", figsize=(7, 4.5),
                       title="Estimator comparison with an outlier").line(label="matheron")
LineGlyph(cressie["lag"].to_numpy(), cressie["semivariance"].to_numpy(), marker="s").line(
    ax=ax, label="cressie (robust)", color="#DC143C")
ax.set_xlabel("lag")
ax.set_ylabel("semivariance")
ax.legend()

## 4. Choosing a model with cross-validation

Don't eyeball it — fit each candidate and let **leave-one-out cross-validation** rank them. The best model
minimizes RMSE while keeping the standardized RMSE near 1 (honest uncertainty). We reuse a clean field here.

In [ ]:
clean = np.sin(xy[:, 0] / 20.0) * np.cos(xy[:, 1] / 20.0) * 8.0 + 20.0
gdf2 = gpd.GeoDataFrame({"z": clean}, geometry=[Point(*p) for p in xy], crs="EPSG:32633")
s2 = Samples(gdf2)

rows = []
for name in ["spherical", "exponential", "gaussian", "matern"]:
    vg = s2.variogram("z").fit(model=name)
    summary = s2.cross_validate("z", vg, n_neighbors=None).attrs["summary"]
    rows.append((name, round(vg.nugget, 3), round(vg.sill, 2), round(vg.range_, 1),
                 round(summary["RMSE"], 4), round(summary["RMSE_standardized"], 3)))

import pandas as pd
pd.DataFrame(rows, columns=["model", "nugget", "sill", "range", "RMSE", "RMSE_std"]).set_index("model")

## 5. When a fit is impossible — `VariogramFitError`

If the field has no spatial structure to fit (e.g. a constant column), fitting raises a clear
`VariogramFitError` rather than returning garbage. Catch it and fall back (e.g. to IDW).

In [ ]:
from geostatista import VariogramFitError

flat = gpd.GeoDataFrame({"z": np.full(len(xy), 5.0)}, geometry=[Point(*p) for p in xy], crs="EPSG:32633")
try:
    Samples(flat).variogram("z").fit(model="spherical")
except VariogramFitError as exc:
    print("caught VariogramFitError:", exc)

**Takeaways:** pick `spherical`/`exponential` for most fields, `gaussian` for very smooth ones, `matern` when you
want to tune smoothness; use the `cressie` estimator when outliers are a concern; and always let cross-validation
arbitrate. Back to the [kriging workflow](01_kriging_workflow.ipynb).